In [1]:
import pickle 
import re

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import CXGate, PauliEvolutionGate
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router import CommutingGateRouter
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian

In [2]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [3]:
filename = 'test_N2_W2'
copy_numbers = [1,1]
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)

full_hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)

ess = ExtendedSwapStrategy.from_all_to_all(n * T)
num_physical_qubits = ess._num_vertices

pm_rz = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            ess,
            max_layers=0,
            perform_extra_swaps=True
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(full_hamiltonian), range(num_physical_qubits))   
tqc_rz = pm_rz.run(qc)

Keeping constraints at times: [0]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator


In [4]:
tqc_rz.data

[CircuitInstruction(operation=Instruction(name='rz', num_qubits=1, num_clbits=0, params=[8.875]), qubits=(<Qubit register=(6, "q"), index=0>,), clbits=()), CircuitInstruction(operation=Instruction(name='cx', num_qubits=2, num_clbits=0, params=[]), qubits=(<Qubit register=(6, "q"), index=2>, <Qubit register=(6, "q"), index=1>), clbits=()), CircuitInstruction(operation=Instruction(name='rz', num_qubits=1, num_clbits=0, params=[0.625]), qubits=(<Qubit register=(6, "q"), index=1>,), clbits=()), CircuitInstruction(operation=Instruction(name='cx', num_qubits=2, num_clbits=0, params=[]), qubits=(<Qubit register=(6, "q"), index=0>, <Qubit register=(6, "q"), index=1>), clbits=()), CircuitInstruction(operation=Instruction(name='rz', num_qubits=1, num_clbits=0, params=[0.625]), qubits=(<Qubit register=(6, "q"), index=1>,), clbits=()), CircuitInstruction(operation=Instruction(name='rz', num_qubits=1, num_clbits=0, params=[-1.125]), qubits=(<Qubit register=(6, "q"), index=3>,), clbits=()), CircuitI

In [5]:
full_hamiltonian

SparsePauliOp(['IIIIII', 'IIIIIZ', 'IIIZZI', 'IIIZZZ', 'IIZIII', 'IIZIIZ', 'IIZZZI', 'IIZZZZ', 'IZIIZI', 'IZIIZZ', 'IZIZII', 'IZIZIZ', 'IZZIZI', 'IZZIZZ', 'IZZZII', 'IZZZIZ', 'ZIIIZI', 'ZIIIZZ', 'ZIIZII', 'ZIIZIZ', 'ZIZIZI', 'ZIZIZZ', 'ZIZZII', 'ZIZZIZ', 'ZZIIII', 'ZZIIIZ', 'ZZIZZI', 'ZZIZZZ', 'ZZZIII', 'ZZZIIZ', 'ZZZZZI', 'ZZZZZZ'],
              coeffs=[ 5.9375+0.j,  4.4375+0.j,  0.3125+0.j,  0.3125+0.j, -0.5625+0.j,
 -0.0625+0.j,  0.3125+0.j,  0.3125+0.j,  0.5625+0.j,  0.5625+0.j,
 -0.3125+0.j, -0.3125+0.j,  0.5625+0.j,  0.5625+0.j, -0.3125+0.j,
 -0.3125+0.j,  0.3125+0.j,  0.3125+0.j, -0.3125+0.j, -0.3125+0.j,
  0.3125+0.j,  0.3125+0.j, -0.3125+0.j, -0.3125+0.j, -0.3125+0.j,
 -0.3125+0.j,  0.3125+0.j,  0.3125+0.j, -0.3125+0.j, -0.3125+0.j,
  0.3125+0.j,  0.3125+0.j])

In [6]:
reordered_hamiltonian = SparsePauliOp.from_sparse_list([], tqc_rz.num_qubits)
for instruction in tqc_rz.data:
    if instruction.operation.name == 'rz':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if not len(matches) == 1:
            raise Exception('Bad Rz')
        reordered_hamiltonian += SparsePauliOp.from_sparse_list([('Z', [int(matches[0])], instruction.operation.params[0] / 2)], tqc_rz.num_qubits)
    if instruction.operation.name == 'PauliEvolution':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        reordered_hamiltonian += SparsePauliOp.from_sparse_list([('Z' * len(matches), [int(x) for x in matches], instruction.operation.params[0])], tqc_rz.num_qubits)
reordered_hamiltonian

SparsePauliOp(['IIIIII', 'IIIIIZ', 'IIIIZI', 'IIIIZI', 'IIZIII', 'IIIIZI', 'IIIIZI', 'IIIIZI', 'IIIIZI', 'IIIIZI', 'IIIIZI', 'IIIIZI', 'IIIIZI', 'IIIIZI', 'IIIIIZ', 'IIIIIZ', 'IIIIIZ', 'IIIIZI', 'IIIIZI', 'IIIZII', 'IIIIIZ', 'IIIZII', 'IIIZII', 'IIIZII', 'IIIZII', 'IIIIZI', 'IZIIII', 'IZIIII', 'IZIIII', 'IZIIII', 'IIIZII', 'IIIZII'],
              coeffs=[ 0.    +0.j,  4.4375+0.j,  0.3125+0.j,  0.3125+0.j, -0.5625+0.j,
  0.3125+0.j,  0.3125+0.j,  0.3125+0.j,  0.3125+0.j,  0.3125+0.j,
  0.3125+0.j,  0.3125+0.j,  0.3125+0.j,  0.3125+0.j, -0.0625+0.j,
  0.5625+0.j,  0.5625+0.j,  0.5625+0.j,  0.5625+0.j, -0.3125+0.j,
 -0.3125+0.j, -0.3125+0.j, -0.3125+0.j, -0.3125+0.j, -0.3125+0.j,
  0.3125+0.j, -0.3125+0.j, -0.3125+0.j, -0.3125+0.j, -0.3125+0.j,
 -0.3125+0.j, -0.3125+0.j])

In [7]:
tqc_rz.draw(fold=-1)

global phase: 0.34569
         ┌───────────┐                                                                                                                                                                                                  ┌───┐┌────────────┐                         ┌───┐┌───────────┐ ┌───┐┌───────────┐┌───┐             ┌───┐┌────────────┐             ┌───┐                                                                                                                                                                                                                                                     
q_0 -> 0 ┤ Rz(8.875) ├────────────────■─────────────────────────────────────────────────────■───────────────────────────────────■─────────────────────────────────────────────────────■─────────────────────────────────┤ X ├┤ Rz(-0.125) ├─────────────────────────┤ X ├┤ Rz(1.125) ├─┤ X ├┤ Rz(1.125) ├┤ X ├─────────────┤ X ├┤ Rz(-0.625) ├─────────────┤ X ├────────────────────────────────────■─────────────────────────────────────■───────────────────────────────────────────────■─────────────────────────────────────■────────────────────────────────────────────────────■──────────────────■────────────
         └───┬───┬───┘ ┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐┌───┐┌───────────┐┌─┴─┐┌───────────┐┌───┐┌───────────┐└─┬─┘└───┬───┬────┘          ┌───┐┌───┐┌───┐└─┬─┘└───────────┘ └─┬─┘└───────────┘└─┬─┘┌───────────┐└─┬─┘└───┬───┬────┘┌───────────┐└─┬─┘┌───┐     ┌───┐                     │                                     │                     ┌───┐┌───────────┐ ┌───┐  │                                     │                                                    │                  │            
q_1 -> 1 ────┤ X ├─────┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├┤ X ├┤ Rz(0.625) ├──┼──────┤ X ├───────────────┤ X ├┤ X ├┤ X ├──■──────────────────┼─────────────────■──┤ Rz(1.125) ├──┼──────┤ X ├─────┤ Rz(1.125) ├──┼──┤ X ├─────┤ X ├─────────────────────┼─────────────────────────────────────┼─────────────────────┤ X ├┤ Rz(0.625) ├─┤ X ├──┼─────────────────────────────────────┼────────────────────────────────────────────────────┼──────────────────┼────────────
             └─┬─┘     └───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘├───┤└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘└─┬─┘└───────────┘└───┘└───────────┘└─┬─┘└───────────┘  │      └─┬─┘     ┌───┐     └─┬─┘└─┬─┘└─┬─┘┌───┐┌────────────┐  │                    └───────────┘  │      └─┬─┘     └───────────┘  │  └─┬─┘┌───┐└─┬─┘┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐┌────────────┐┌─┴─┐┌────────────┐┌───┐└─┬─┘└───────────┘ └─┬─┘  │                                     │                     ┌───┐     ┌───┐┌────────────┐┌─┴─┐┌────────────┐┌─┴─┐┌───┐     
q_2 -> 2 ──────■────────────────────────────────────────┼─────────────────┼───────────────┤ X ├───────────────┼───────────────────────────────────■─────────────────┼───────────────────────────────────┼─────────────────┼────────■───────┤ X ├───────┼────■────┼──┤ X ├┤ Rz(-0.625) ├──┼───────────────────────────────────■────────┼──────────────────────■────┼──┤ X ├──┼──┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├──■──────────────────■────┼─────────────────────────────────────┼─────────────────────┤ X ├─────┤ X ├┤ Rz(-0.625) ├┤ X ├┤ Rz(-0.625) ├┤ X ├┤ X ├─────
         ┌────────────┐                                 │                 │               └─┬─┘               │                                                     │                                   │                 │                └─┬─┘       │         │  └─┬─┘└────────────┘  │                                            │                           │  └─┬─┘  │  └─┬─┘└────────────┘└───

In [8]:
hubo_hams = {}
hubo_circs = {}
hubo_reordered_hams = {}
results = {}
for name, filename, copy_numbers in zip(
    ['H_6', 'H_9', 'H_12', 'H_20(1)', 'H_20(2)', 'H_30'],
    ['test_N2_W2', 'trivial', 'test_N3_W4', 'test_N4_W5', 'test_N7_W5', 'test_N10_W6'], 
    [[1,1], [1,1,1], [2,1,1], [2,1,1,1], [1,1,1,0,1,0,1], [1,1,0,0,1,0,1,1,0,1]]
):
    filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
    graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)

    full_hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
    hubo_hams[name] = full_hamiltonian
    
    ess = ExtendedSwapStrategy.from_all_to_all(n * T)
    num_physical_qubits = ess._num_vertices
    
    pm_rz = PassManager(
        [
            FindCommutingPauliEvolutionsMulti(), 
            CommutingGateRouter(
                ess,
                max_layers=0,
                perform_extra_swaps=True
            ),
            SwapToFinalMapping(),
            InverseCancellation(gates_to_cancel=[CXGate()]),
            CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
            InverseCancellation(gates_to_cancel=[CXGate()]),
        ]
    )
    qc = QuantumCircuit(num_physical_qubits)
    qc.append(PauliEvolutionGate(full_hamiltonian), range(num_physical_qubits))   
    tqc_rz = pm_rz.run(qc)
    print_circuit_info(tqc_rz, 'Rz circuit')
    hubo_circs[name] = tqc_rz
    
    reordered_hamiltonian = SparsePauliOp.from_sparse_list([], tqc_rz.num_qubits)
    for instruction in tqc_rz.data:
        if instruction.operation.name == 'rz':
            qubits_str = str(instruction.qubits)
            matches = re.findall('index=([0-9]+)', qubits_str)
            if not len(matches) == 1:
                raise Exception('Bad Rz')
            reordered_hamiltonian += SparsePauliOp.from_sparse_list([('Z', [int(matches[0])], instruction.operation.params[0] / 2)], tqc_rz.num_qubits)
        if instruction.operation.name == 'PauliEvolution':
            qubits_str = str(instruction.qubits)
            matches = re.findall('index=([0-9]+)', qubits_str)
            reordered_hamiltonian += SparsePauliOp.from_sparse_list([('Z' * len(matches), [int(x) for x in matches], instruction.operation.params[0])], tqc_rz.num_qubits)
    hubo_reordered_hams[name] = reordered_hamiltonian


Keeping constraints at times: [0]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 6 qubits
Rz circuit has 52 non-local gates and 40 non-local depth
Rz circuit contains ['cx', 'rz'] gates.
Keeping constraints at times: [0 1]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 9 qubits
Rz circuit has 154 non-local gates and 114 non-local depth
Rz circuit contains ['cx', 'rz'] gates.
Keeping constraints at times: [1 2 0]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 12 qubits
Rz circuit has 228 non-local gates and 120 non-local depth
Rz circuit contains ['cx', 'rz'] gates.
Keeping constraints at times: [3 0 2 1]
Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Rz circuit has 20 qubits
Rz circuit has 1188

In [ ]:
from qiskit.qasm3 import dump as qasm3_dump
for name in hubo_circs.keys():
    with open(f'../out/ionq/qasm3/hubo_circuit_{name}.txt', 'w') as f:
        qasm3_dump(hubo_circs[name], f)

In [ ]:
from qiskit import qasm2
for name in hubo_circs.keys():
    with open(f'../out/ionq/qasm2/hubo_circuit_{name}.txt', 'w') as f:
        qasm2.dump(hubo_circs[name], f)

In [ ]:
with open('../out/ionq/hubo_hamiltonians.pkl', 'wb') as f:
    pickle.dump(hubo_hams, f)

In [9]:
with open('../out/ionq/hubo_reordered_hamiltonians.pkl', 'wb') as f:
    pickle.dump(hubo_reordered_hams, f)

In [ ]:
from qiskit_qaoa.utils.hamiltonian_utils import get_objective_and_hamiltonian
filename = 'test_N3_W4'
data_file = f'/lustre/scratch127/qpg/jc59/out/tangle/qubo_data_{filename}.gfa.pkl'

_, hamiltonian, _ = get_objective_and_hamiltonian(data_file)

In [ ]:
qubo_hams = {}

for name, filename in zip(
    ['Q_6', 'Q_12', 'Q_16', 'Q_30', 'Q_35'],
    ['test_N2_W2', 'trivial', 'test_N3_W4', 'test_N4_W5', 'test_N4_W6'], 
):
    data_file = f'/lustre/scratch127/qpg/jc59/out/tangle/qubo_data_{filename}.gfa.pkl'
    _, hamiltonian, _ = get_objective_and_hamiltonian(data_file)
    qubo_hams[name] = hamiltonian

In [ ]:
for h in qubo_hams.values():
    print(h.num_qubits)

In [ ]:
import pickle 
with open('../out/ionq/qubo_hamiltonians.pkl', 'wb') as f:
    pickle.dump(qubo_hams, f)

In [ ]:
%pip list

In [ ]:
router = CommutingGateRouter(
    ess,
    max_layers=0,
    perform_extra_swaps=True
)

In [ ]:
router._compute_site_abstract(
    (0,1,2),
    0,
    
)